In [1]:
import json
import pandas as pd
import pandas as pd
from pathlib import Path
from typing import Any, List

def read_jsonl(path: Path):
    with path.open(encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def merge_runs(items, merge_labels=("equivalent", "contradiction"), sep=" "):
    out = []
    buf_label = None
    buf_span1, buf_span2 = [], []

    def _join(parts, sep_):
        s = sep_.join(p for p in parts if p is not None and p != "")
        return " ".join(s.split())

    def flush():
        nonlocal buf_label, buf_span1, buf_span2
        if buf_label is not None:
            out.append({
                "span_1": _join(buf_span1, sep),
                "span_2": _join(buf_span2, sep),
                "label": buf_label,
            })
            buf_label = None
            buf_span1, buf_span2 = [], []

    for it in items:
        lab = it.get("label")
        if lab in merge_labels:
            if buf_label is None:
                buf_label = lab
                buf_span1.append(it.get("span_1", ""))
                buf_span2.append(it.get("span_2", ""))
            elif buf_label == lab:
                buf_span1.append(it.get("span_1", ""))
                buf_span2.append(it.get("span_2", ""))
            else:
                flush()
                buf_label = lab
                buf_span1.append(it.get("span_1", ""))
                buf_span2.append(it.get("span_2", ""))
        else:
            flush()
            out.append(it)

    flush()
    return out

In [2]:
claims = pd.read_csv('inference_v2/mmbert_sft_on_rules_paraphrased (4).csv')[['url', 'paragraph_2', 'paraphrased_paragraph_1', 'output']]
data = pd.read_csv('rules_paraphrased_gpt4o.csv')[['url', 'paraphrased_paragraph_1', 'paragraph_2', 'paraphrased_output']].dropna()

claims['output'] = claims['output'].fillna("[]").apply(eval).apply(merge_runs)

aggregated_data = data.merge(claims, on=['url', 'paraphrased_paragraph_1', 'paragraph_2'])
aggregated_data['paraphrased_output'] = aggregated_data['paraphrased_output'].apply(eval)

aggregated_data['paraphrased_output'] = aggregated_data['paraphrased_output'].apply(merge_runs)

In [3]:
aggregated_data['span_1'] = aggregated_data['paraphrased_output'].apply(lambda x: [s['span_1'] for s in x])
aggregated_data['span_2'] = aggregated_data['paraphrased_output'].apply(lambda x: [s['span_2'] for s in x])
aggregated_data['Label'] = aggregated_data['paraphrased_output'].apply(lambda x: [s['label'] for s in x])

aggregated_data['Label'] = aggregated_data['Label'].apply(lambda x: [a.lower() for a in x])

In [4]:
anchors = []
for i, row in aggregated_data.iterrows():
    tmp = []
    for i in row['paraphrased_output']:
        if i['label'] == 'addition':
            tmp.append(i['anchor'])
        else:
            tmp.append('')
    anchors.append(tmp)

aggregated_data['Anchor_span'] = anchors

In [5]:
import numpy as np
from tqdm import tqdm
import pandas as pd
from collections import Counter
from rapidfuzz import fuzz, process

def fill_none(s):
    if not s:
        return ''
    return s

from difflib import SequenceMatcher

def char_match_percent(s1: str, s2: str) -> float:
    return SequenceMatcher(None, s1, s2).ratio() * 100

fn_eq = [
    1
    if isinstance(p['span_1'], str)
       and fill_none(p['span_1']).strip() == fill_none(p['span_2']).strip()
       and p['label'].lower() != 'equivalent'
    else 0
    for preds in aggregated_data['output']
    for p in preds
]
print('Global FN‑equivalent rate:', round(np.mean(fn_eq) * 100, 2), '%')

for col in list(aggregated_data.filter(like='F1_')):
    del aggregated_data[col]

thresholds = (0, 25, 50, 75, 80, 85, 90, 95, 99)
results = []

for TH in tqdm(thresholds):
    anchor_hit = anchor_total = 0
    cnt = Counter()
    confusion = Counter()
    per_sample_f1 = []

    cont_span_length = {'include': [], 'percent_equivalent': []}

    for idx, row in aggregated_data.iterrows():
        try:
            gold = []
            gold = [
                {'span_1': s1, 'span_2': s2, 'label': l.lower(), 'anchor': a}
                for s1, s2, l, a in zip(row['span_1'], row['span_2'],
                                         row['Label'], row['Anchor_span'])
            ]
            pred = [{**p, 'label': p['label'].lower()} for p in row['output']]
    
            pred_s1 = [p['span_1'].strip() if isinstance(p['span_1'], str) else ''
                       for p in pred]
            pred_s2 = [p['span_2'].strip() if isinstance(p['span_2'], str) else ''
                       for p in pred]
    
            matched = set()
            loc_tp = loc_fp = loc_fn = 0
    
            for g in gold:
                if g['label'] == 'addition':
                    print(g)
    
                g1 = g['span_1'].strip() if isinstance(g['span_1'], str) else ''
                g2 = g['span_2'].strip() if isinstance(g['span_2'], str) else ''
    
                idx1 = process.cdist([g1], pred_s1).argmax()
                idx2 = process.cdist([g2], pred_s2, scorer=fuzz.ratio).argmax()
    
                match_s1 = fuzz.ratio(g1, pred_s1[idx1]) > TH
                match_s2 = fuzz.ratio(g2, pred_s2[idx2]) > TH
    
                if match_s1 and match_s2 and idx1 == idx2:
                    p_label = pred[idx1]['label']
                    if g['label'] == p_label:
                        cnt['TP_'+g['label']] += 1
                        matched.add(idx1)
                        loc_tp += 1

                        if p_label == 'contradiction':
                            if pred_s1[idx1] in g1:
                                cont_span_length['include'].append(True)
                            else:
                                cont_span_length['include'].append(False)
                            if pred_s2[idx2] in g2:
                                cont_span_length['include'].append(True)
                            else:
                                cont_span_length['include'].append(False)
    
                            cont_span_length['percent_equivalent'].append(char_match_percent(s1=pred_s1[idx1], s2=g1))
                            cont_span_length['percent_equivalent'].append(char_match_percent(s1=pred_s2[idx2], s2=g2))
                    else:
                        confusion[(g['label'], p_label)] += 1
                        cnt['FN_'+g['label']] += 1
                        loc_fn += 1
                else:
                    cnt['FN_'+g['label']] += 1
                    loc_fn += 1
    
                # anchor как было
                if g['label'] == 'addition':
                    ga = g.get('anchor'); pa = pred[idx1].get('anchor')
                    if isinstance(ga, str) and isinstance(pa, str) and ga and pa:
                        anchor_total += 1
                        if fuzz.ratio(ga.strip(), pa.strip()) > TH:
                            anchor_hit += 1
        except Exception as E:
            print(E)
        # FP
        for j, p in enumerate(pred):
            if j in matched or p['label'] not in ('contradiction', 'addition'):
                continue
            cnt['FP_'+p['label']] += 1
            loc_fp += 1

        prec = loc_tp / (loc_tp + loc_fp) if loc_tp + loc_fp else 0
        rec  = loc_tp / (loc_tp + loc_fn) if loc_tp + loc_fn else 0
        f1_doc = 2*prec*rec / (prec + rec) if prec + rec else 0
        per_sample_f1.append(f1_doc)

    aggregated_data[f'F1_{TH}'] = per_sample_f1

    row = {'threshold': TH}
    micro_tp = micro_fp = micro_fn = 0
    for lbl in ('contradiction', 'addition', 'equivalent'):
        tp = cnt['TP_'+lbl]; fp = cnt['FP_'+lbl]; fn = cnt['FN_'+lbl]
        prec = tp/(tp+fp) if tp+fp else 0
        rec  = tp/(tp+fn) if tp+fn else 0
        f1   = 2*prec*rec/(prec+rec) if prec+rec else 0
        row[f'P_{lbl}']  = round(prec,4)
        row[f'R_{lbl}']  = round(rec,4)
        row[f'F1_{lbl}'] = round(f1,4)
        micro_tp += tp; micro_fp += fp; micro_fn += fn

        row['contradiction_span_include'] = np.mean(cont_span_length['include'])
        row['contradiction_span_percent_of_equivalent'] = np.mean(cont_span_length['percent_equivalent'])

    micro_prec = micro_tp/(micro_tp+micro_fp) if micro_tp+micro_fp else 0
    micro_rec  = micro_tp/(micro_tp+micro_fn) if micro_tp+micro_fn else 0
    row['micro_F1'] = round(2*micro_prec*micro_rec/(micro_prec+micro_rec) if micro_prec+micro_rec else 0,3)
    row['anchor_match_rate'] = round(anchor_hit/anchor_total,3) if anchor_total else None

    row['TP_contradiction'] = cnt['TP_contradiction']
    row['TP_addition']    = cnt['TP_addition']
    row['conf_c2a']       = confusion[('contradiction','addition')]
    row['conf_a2c']       = confusion[('addition','contradiction')]

    row['conf_c2a'] = confusion[('contradiction','addition')]
    row['conf_a2c'] = confusion[('addition','contradiction')]

    results.append(row)

results = pd.DataFrame(results)
print(results.to_string(index=False))

for TH in thresholds:
    sub = results[results['threshold']==TH].iloc[0]
    mat = pd.DataFrame(
        [[sub['TP_contradiction'], sub['conf_c2a']],
         [sub['conf_a2c'],        sub['TP_addition']]],
        index=['gold_contradiction','gold_addition'],
        columns=['pred_contradiction','pred_addition']
    )
    print(f"\nThreshold = {TH}% confusion matrix:")
    print(mat.to_string())


Global FN‑equivalent rate: 0.97 %


 33%|██████████████████████████████████████████████████████████████████████                                                                                                                                            | 3/9 [00:00<00:00, 29.64it/s]

{'span_1': 'Безвозмездная передача иными лицами имущества в личный фонд не допускается.', 'span_2': '', 'label': 'addition', 'anchor': ''}
{'span_1': '(если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет', 'span_2': '', 'label': 'addition', 'anchor': ''}
{'span_1': '(если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет', 'span_2': '', 'label': 'addition', 'anchor': ''}
{'span_1': 'При этом налоговый орган по запросу иностранного гражданина, лица без гражданства, поставленных на учет в налоговом органе в соответствии с пунктом 7.4 статьи 83 настоящего Кодекса, обязан представить им указанное уведомление в письменной форме на бумажном носителе.', 'span_2': '', 'label': 'addition', 'anchor': ''}
{'span_1': 'При этом налоговый орган обязан представить предусмотренные настоящим пунктом документы в письменной форме на бумажном носителе по запросу организации или физического лица, в том числе индивидуального предпринимателя.

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 44.22it/s]

 threshold  P_contradiction  R_contradiction  F1_contradiction  contradiction_span_include  contradiction_span_percent_of_equivalent  P_addition  R_addition  F1_addition  P_equivalent  R_equivalent  F1_equivalent  micro_F1 anchor_match_rate  TP_contradiction  TP_addition  conf_c2a  conf_a2c
         0           0.6571           0.5897            0.6216                    0.434783                                 71.694962      0.3810      0.4211         0.40           1.0        0.8205         0.9014     0.742              None                23            8         0         0
        25           0.6176           0.5385            0.5753                    0.476190                                 75.208325      0.3810      0.4211         0.40           1.0        0.7949         0.8857     0.719              None                21            8         0         0
        50           0.4706           0.4103            0.4384                    0.625000                                 8

In [6]:
results[['threshold', 'F1_contradiction', 'F1_addition', 'F1_equivalent', 'micro_F1', 'contradiction_span_include', 'contradiction_span_percent_of_equivalent', 'anchor_match_rate']]

,threshold,F1_contradiction,F1_addition,F1_equivalent,micro_F1,contradiction_span_include,contradiction_span_percent_of_equivalent,anchor_match_rate
0,0,0.6216,0.40,0.9014,0.742,0.434783,71.694962,None
1,25,0.5753,0.40,0.8857,0.719,0.476190,75.208325,None
2,50,0.4384,0.40,0.7907,0.620,0.625000,85.232899,None
3,75,0.2466,0.25,0.4706,0.353,0.777778,96.882074,None
4,80,0.2466,0.25,0.3918,0.314,0.777778,96.882074,None
5,85,0.2192,0.25,0.3404,0.280,0.750000,97.571324,None
6,90,0.1918,0.25,0.3043,0.254,0.714286,98.756937,None
7,95,0.1644,0.25,0.2667,0.227,0.750000,99.248084,None
8,99,0.1096,0.20,0.2273,0.179,1.000000,100.000000,None


In [7]:
results[['threshold', 
         'F1_contradiction', 
         'F1_addition', 
         'micro_F1']].to_excel('excel_metrics/gpt4o_paraphrased_rules_gpt4o.xlsx', index=False)